# 🌍 Global PPTX Scraper — All 72 Countries in One Notebook
Downloads academic .pptx/.ppt files from 72 countries into a **single Google Drive folder**.
Includes dedup, 2MB minimum, Chinese/USA content filtering, and magic byte validation.

In [ ]:
# Step 1: Mount Google Drive & Install deps
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/PPTX_Global_Download'
SEEN_FILE = os.path.join(DRIVE_DIR, 'seen_tags.txt')
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Output: {DRIVE_DIR}')

!pip install -q duckduckgo-search beautifulsoup4 lxml requests

In [ ]:
# Step 2: All 72 Countries — Domains & Topics
COUNTRIES = {
  # AFRICA
  'south_africa': {'tld':'za','domains':['site:uct.ac.za','site:wits.ac.za','site:sun.ac.za','site:up.ac.za','site:ukzn.ac.za','site:nwu.ac.za','site:uj.ac.za','site:unisa.ac.za']},
  'nigeria': {'tld':'ng','domains':['site:unilag.edu.ng','site:ui.edu.ng','site:unn.edu.ng','site:abu.edu.ng','site:oauife.edu.ng']},
  'egypt': {'tld':'eg','domains':['site:cu.edu.eg','site:asu.edu.eg','site:aun.edu.eg','site:mans.edu.eg','site:alexu.edu.eg']},
  'kenya': {'tld':'ke','domains':['site:uonbi.ac.ke','site:ku.ac.ke','site:egerton.ac.ke','site:jkuat.ac.ke']},
  'morocco': {'tld':'ma','domains':['site:um5.ac.ma','site:um6p.ma','site:uca.ac.ma','site:uiz.ac.ma']},
  'ghana': {'tld':'gh','domains':['site:ug.edu.gh','site:knust.edu.gh','site:ucc.edu.gh','site:uew.edu.gh']},
  'tunisia': {'tld':'tn','domains':['site:ucar.tn','site:enit.tn','site:isi.tn','site:fst.tn']},
  # ASIA
  'india': {'tld':'in','domains':['site:iitb.ac.in','site:iitd.ac.in','site:iitm.ac.in','site:iisc.ac.in','site:iitk.ac.in','site:iitkgp.ac.in','site:du.ac.in','site:jnu.ac.in']},
  'japan': {'tld':'jp','domains':['site:u-tokyo.ac.jp','site:kyoto-u.ac.jp','site:osaka-u.ac.jp','site:tohoku.ac.jp','site:titech.ac.jp','site:nii.ac.jp']},
  'south_korea': {'tld':'kr','domains':['site:snu.ac.kr','site:kaist.ac.kr','site:postech.ac.kr','site:korea.ac.kr','site:yonsei.ac.kr']},
  'israel': {'tld':'il','domains':['site:tau.ac.il','site:huji.ac.il','site:technion.ac.il','site:bgu.ac.il','site:weizmann.ac.il']},
  'saudi_arabia': {'tld':'sa','domains':['site:ksu.edu.sa','site:kau.edu.sa','site:kfupm.edu.sa','site:kaust.edu.sa']},
  'thailand': {'tld':'th','domains':['site:chula.ac.th','site:mahidol.ac.th','site:ku.ac.th','site:kmutt.ac.th']},
  'vietnam': {'tld':'vn','domains':['site:vnu.edu.vn','site:hust.edu.vn','site:hcmut.edu.vn','site:fpt.edu.vn']},
  'singapore': {'tld':'sg','domains':['site:nus.edu.sg','site:ntu.edu.sg','site:smu.edu.sg','site:sutd.edu.sg']},
  'malaysia': {'tld':'my','domains':['site:um.edu.my','site:ukm.edu.my','site:usm.my','site:upm.edu.my','site:utm.my']},
  'turkey': {'tld':'tr','domains':['site:metu.edu.tr','site:boun.edu.tr','site:itu.edu.tr','site:bilkent.edu.tr']},
  'indonesia': {'tld':'id','domains':['site:ui.ac.id','site:itb.ac.id','site:ugm.ac.id','site:its.ac.id']},
  'philippines': {'tld':'ph','domains':['site:up.edu.ph','site:ateneo.edu','site:dlsu.edu.ph','site:ust.edu.ph']},
  'pakistan': {'tld':'pk','domains':['site:nust.edu.pk','site:lums.edu.pk','site:uet.edu.pk','site:quaid.edu.pk']},
  'bangladesh': {'tld':'bd','domains':['site:buet.ac.bd','site:du.ac.bd','site:bracu.ac.bd']},
  'uae': {'tld':'ae','domains':['site:ku.ac.ae','site:uaeu.ac.ae','site:aus.edu','site:masdar.ac.ae']},
  'iran': {'tld':'ir','domains':['site:ut.ac.ir','site:sharif.edu','site:iust.ac.ir','site:aut.ac.ir']},
  'jordan': {'tld':'jo','domains':['site:ju.edu.jo','site:just.edu.jo','site:hu.edu.jo']},
  'lebanon': {'tld':'lb','domains':['site:aub.edu.lb','site:lau.edu.lb','site:ul.edu.lb']},
  'qatar': {'tld':'qa','domains':['site:qu.edu.qa','site:hbku.edu.qa','site:qatar.tamu.edu']},
  'sri_lanka': {'tld':'lk','domains':['site:cmb.ac.lk','site:pdn.ac.lk','site:mrt.ac.lk']},
  'kazakhstan': {'tld':'kz','domains':['site:nu.edu.kz','site:kaznu.kz','site:satbayev.university']},
  # EUROPE
  'uk': {'tld':'uk','domains':['site:ox.ac.uk','site:cam.ac.uk','site:ucl.ac.uk','site:imperial.ac.uk','site:ed.ac.uk','site:kcl.ac.uk','site:manchester.ac.uk']},
  'france': {'tld':'fr','domains':['site:univ-paris1.fr','site:polytechnique.fr','site:ens.fr','site:inria.fr','site:cnrs.fr']},
  'germany': {'tld':'de','domains':['site:tu-berlin.de','site:tu-muenchen.de','site:lmu.de','site:uni-heidelberg.de','site:kit.edu']},
  'italy': {'tld':'it','domains':['site:uniroma1.it','site:polimi.it','site:unibo.it','site:unipd.it','site:unimi.it']},
  'spain': {'tld':'es','domains':['site:ub.edu','site:uam.es','site:ucm.es','site:upc.edu','site:upm.es']},
  'netherlands': {'tld':'nl','domains':['site:tudelft.nl','site:uva.nl','site:uu.nl','site:leidenuniv.nl','site:tue.nl']},
  'switzerland': {'tld':'ch','domains':['site:ethz.ch','site:epfl.ch','site:uzh.ch','site:unibe.ch']},
  'belgium': {'tld':'be','domains':['site:kuleuven.be','site:ugent.be','site:uclouvain.be','site:ulb.be']},
  'austria': {'tld':'at','domains':['site:univie.ac.at','site:tuwien.ac.at','site:tugraz.at','site:uibk.ac.at']},
  'norway': {'tld':'no','domains':['site:uio.no','site:ntnu.no','site:uib.no','site:uit.no']},
  'denmark': {'tld':'dk','domains':['site:ku.dk','site:dtu.dk','site:au.dk','site:sdu.dk']},
  'finland': {'tld':'fi','domains':['site:aalto.fi','site:helsinki.fi','site:tuni.fi','site:oulu.fi']},
  'ireland': {'tld':'ie','domains':['site:tcd.ie','site:ucd.ie','site:nuigalway.ie','site:ucc.ie']},
  'portugal': {'tld':'pt','domains':['site:up.pt','site:tecnico.ulisboa.pt','site:uc.pt','site:uminho.pt']},
  'greece': {'tld':'gr','domains':['site:uoa.gr','site:auth.gr','site:ntua.gr','site:upatras.gr']},
  'poland': {'tld':'pl','domains':['site:uw.edu.pl','site:agh.edu.pl','site:put.poznan.pl','site:pw.edu.pl']},
  'czech_republic': {'tld':'cz','domains':['site:cuni.cz','site:cvut.cz','site:muni.cz','site:vutbr.cz']},
  'hungary': {'tld':'hu','domains':['site:elte.hu','site:bme.hu','site:u-szeged.hu','site:unideb.hu']},
  'romania': {'tld':'ro','domains':['site:unibuc.ro','site:ubbcluj.ro','site:upb.ro','site:uaic.ro']},
  'ukraine': {'tld':'ua','domains':['site:knu.ua','site:kpi.ua','site:lnu.edu.ua','site:oneu.edu.ua']},
  'croatia': {'tld':'hr','domains':['site:unizg.hr','site:unist.hr','site:uniri.hr']},
  'serbia': {'tld':'rs','domains':['site:bg.ac.rs','site:uns.ac.rs','site:ni.ac.rs']},
  'russia': {'tld':'ru','domains':['site:msu.ru','site:spbu.ru','site:mipt.ru','site:nsu.ru']},
  'bulgaria': {'tld':'bg','domains':['site:uni-sofia.bg','site:tu-sofia.bg','site:tu-plovdiv.bg']},
  'slovakia': {'tld':'sk','domains':['site:uniba.sk','site:tuke.sk','site:stuba.sk']},
  'lithuania': {'tld':'lt','domains':['site:vu.lt','site:ktu.lt','site:vgtu.lt']},
  'slovenia': {'tld':'si','domains':['site:uni-lj.si','site:um.si','site:upr.si']},
  'estonia': {'tld':'ee','domains':['site:ut.ee','site:taltech.ee','site:tlu.ee']},
  'nordics': {'tld':'se','domains':['site:kth.se','site:uu.se','site:lu.se','site:chalmers.se','site:gu.se']},
  # AMERICAS
  'canada': {'tld':'ca','domains':['site:utoronto.ca','site:ubc.ca','site:mcgill.ca','site:uwaterloo.ca','site:ualberta.ca']},
  'mexico': {'tld':'mx','domains':['site:unam.mx','site:tec.mx','site:ipn.mx','site:udg.mx']},
  'brazil': {'tld':'br','domains':['site:usp.br','site:unicamp.br','site:ufrj.br','site:ufmg.br','site:ufrgs.br']},
  'argentina': {'tld':'ar','domains':['site:uba.ar','site:unlp.edu.ar','site:unc.edu.ar','site:itba.edu.ar']},
  'chile': {'tld':'cl','domains':['site:uchile.cl','site:uc.cl','site:usach.cl','site:udec.cl']},
  'colombia': {'tld':'co','domains':['site:unal.edu.co','site:uniandes.edu.co','site:javeriana.edu.co']},
  'peru': {'tld':'pe','domains':['site:pucp.edu.pe','site:uni.edu.pe','site:unmsm.edu.pe']},
  'ecuador': {'tld':'ec','domains':['site:epn.edu.ec','site:uce.edu.ec','site:puce.edu.ec']},
  'venezuela': {'tld':'ve','domains':['site:ucv.ve','site:usb.ve','site:ula.ve']},
  'costa_rica': {'tld':'cr','domains':['site:ucr.ac.cr','site:tec.ac.cr','site:una.ac.cr']},
  'uruguay': {'tld':'uy','domains':['site:udelar.edu.uy','site:ort.edu.uy']},
  'cuba': {'tld':'cu','domains':['site:uh.cu','site:uho.edu.cu','site:cujae.edu.cu']},
  # OCEANIA
  'australia': {'tld':'au','domains':['site:usyd.edu.au','site:unimelb.edu.au','site:anu.edu.au','site:unsw.edu.au','site:uq.edu.au']},
  'new_zealand': {'tld':'nz','domains':['site:auckland.ac.nz','site:otago.ac.nz','site:canterbury.ac.nz','site:waikato.ac.nz']},
}

TOPICS = [
  'lecture slides','presentation','course materials','seminar','workshop',
  'computer science','artificial intelligence','machine learning','deep learning',
  'data science','engineering','mechanical engineering','electrical engineering',
  'civil engineering','physics','quantum mechanics','thermodynamics',
  'chemistry','organic chemistry','biochemistry','biology','genetics',
  'biotechnology','mathematics','statistics','calculus','linear algebra',
  'medicine','anatomy','pharmacology','epidemiology',
  'economics','business','finance','marketing','management',
  'history','philosophy','psychology','sociology','political science',
  'architecture','environmental science','climate change','renewable energy',
  'robotics','cybersecurity','networking','database systems','operating systems',
]

print(f'✅ {len(COUNTRIES)} countries, {len(TOPICS)} topics configured')

In [ ]:
# Step 3: Core Scraper Engine
import hashlib, logging, os, random, re, time, warnings
from pathlib import Path
from urllib.parse import urljoin, urlparse
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

try:
    from duckduckgo_search import DDGS
    from duckduckgo_search.exceptions import RatelimitException
except ImportError:
    from ddgs import DDGS
    try:
        from ddgs.exceptions import RatelimitException
    except: RatelimitException = Exception

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('GlobalScraper')

PRES_RE = re.compile(r'\.pptx?($|[?#&\s])', re.IGNORECASE)
RAW_RE = re.compile(r'https?://[^"\'\s>{}\[\]\\]+\.pptx?', re.IGNORECASE)

def load_seen_tags():
    tags = set()
    if os.path.exists(SEEN_FILE):
        with open(SEEN_FILE) as f:
            tags = {l.strip() for l in f if l.strip()}
    # Also scan existing files in Drive
    for fn in os.listdir(DRIVE_DIR):
        if fn.endswith(('.ppt','.pptx')) and '_' in fn:
            tag = fn.split('_')[0]
            if len(tag) == 10: tags.add(tag)
    return tags

def save_tag(tag):
    with open(SEEN_FILE, 'a') as f:
        f.write(tag + '\n')

class GlobalScraper:
    def __init__(self):
        self.session = requests.Session()
        retry = Retry(total=3, backoff_factor=0.5, status_forcelist=[429,500,502,503,504])
        self.session.mount('https://', HTTPAdapter(max_retries=retry))
        self.session.mount('http://', HTTPAdapter(max_retries=retry))
        self.session.headers.update({'User-Agent':'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120.0 Safari/537.36'})
        self.seen_urls = set()
        self.seen_tags = load_seen_tags()
        self.count = 0
        logger.info(f'Loaded {len(self.seen_tags)} previously seen tags')

    def search_ddgs(self, query):
        found = []
        for attempt in range(3):
            try:
                with DDGS() as ddgs:
                    results = list(ddgs.text(query, max_results=50))
                for r in results:
                    if r.get('href'): found.append(r['href'])
                break
            except RatelimitException:
                logger.warning('DuckDuckGo rate-limited, trying Bing...')
                return self.search_bing(query)
            except Exception as e:
                if attempt < 2: time.sleep(5)
                else: break
        return found

    def search_bing(self, query):
        found = []
        try:
            url = f'https://www.bing.com/search?q={requests.utils.quote(query)}&count=50'
            resp = self.session.get(url, timeout=15)
            if resp.ok:
                soup = BeautifulSoup(resp.text, 'lxml')
                for a in soup.find_all('a', href=True):
                    href = a['href']
                    if PRES_RE.search(href) and href.startswith('http'):
                        found.append(href)
        except: pass
        return found

    def extract_from_page(self, url):
        try:
            resp = self.session.get(url, timeout=30)
            if not resp.ok: return []
            found = []
            for m in RAW_RE.finditer(resp.text): found.append(m.group(0))
            soup = BeautifulSoup(resp.text, 'lxml')
            for a in soup.find_all('a', href=True):
                if PRES_RE.search(a['href']):
                    found.append(urljoin(url, a['href']))
            return list(set(found))
        except: return []

    def download(self, url):
        tag = hashlib.sha1(url.encode()).hexdigest()[:10]
        if tag in self.seen_tags: return False
        name = Path(urlparse(url).path).name or 'file.pptx'
        if not name.lower().endswith(('.pptx','.ppt')): name += '.pptx'
        clean = re.sub(r'[^\w.\-]', '_', name)
        fname = f'{tag}_{clean}'
        dest = os.path.join(DRIVE_DIR, fname)
        if os.path.exists(dest): self.seen_tags.add(tag); return False
        try:
            # HEAD check for size
            try:
                head = self.session.head(url, timeout=10, allow_redirects=True)
                cl = int(head.headers.get('Content-Length', 0))
                if 0 < cl < 2097152: return False
            except: pass
            resp = self.session.get(url, timeout=45, stream=True)
            if not resp.ok: return False
            if 'text/html' in resp.headers.get('Content-Type','').lower(): return False
            with open(dest, 'wb') as f:
                for chunk in resp.iter_content(65536): f.write(chunk)
            sz = os.path.getsize(dest)
            if sz < 2097152:
                os.remove(dest); return False
            # Magic byte check
            with open(dest, 'rb') as f:
                hdr = f.read(8)
            if hdr[:2] != b'PK' and hdr[:8] != b'\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1':
                os.remove(dest); return False
            self.seen_tags.add(tag)
            save_tag(tag)
            self.count += 1
            logger.info(f'  ✅ [{self.count}] {fname} ({sz//1024}KB)')
            return True
        except:
            if os.path.exists(dest): os.remove(dest)
            return False

    def scrape_country(self, name, config, target_per_country=200):
        domains = config['domains']
        local_count = 0
        queries = []
        # Build queries
        for domain in domains:
            queries.append(f'filetype:pptx {domain}')
            queries.append(f'filetype:ppt {domain}')
        for topic in random.sample(TOPICS, min(20, len(TOPICS))):
            for domain in random.sample(domains, min(3, len(domains))):
                queries.append(f'\"{topic}\" filetype:pptx {domain}')
        random.shuffle(queries)
        
        for qi, query in enumerate(queries, 1):
            if local_count >= target_per_country: break
            urls = self.search_ddgs(query)
            for url in urls:
                if local_count >= target_per_country: break
                if url in self.seen_urls: continue
                self.seen_urls.add(url)
                to_dl = []
                if PRES_RE.search(url):
                    to_dl.append(url)
                else:
                    to_dl.extend(self.extract_from_page(url))
                for dl_url in to_dl:
                    dl_tag = hashlib.sha1(dl_url.encode()).hexdigest()[:10]
                    if dl_tag in self.seen_tags: continue
                    if self.download(dl_url):
                        local_count += 1
            time.sleep(2.0 + random.uniform(0.1, 0.5))
        return local_count

print('✅ GlobalScraper engine ready')

In [ ]:
# Step 4: Run All 72 Countries!
from IPython.display import clear_output
import json
from datetime import datetime

TARGET_PER_COUNTRY = 200  # Adjust as needed
scraper = GlobalScraper()
results = {}

country_list = list(COUNTRIES.items())
total_countries = len(country_list)

for ci, (country, config) in enumerate(country_list, 1):
    clear_output(wait=True)
    display_name = country.replace('_', ' ').title()
    print('='*60)
    print(f'🌍 [{ci}/{total_countries}] Scraping: {display_name}')
    print(f'   Domains: {len(config["domains"])} | Target: {TARGET_PER_COUNTRY}')
    print(f'   Total downloaded so far: {scraper.count}')
    print(f'   Completed: {json.dumps({k.replace("_"," ").title():v for k,v in results.items()}, indent=2) if results else "None yet"}')
    print('='*60)
    
    got = scraper.scrape_country(country, config, TARGET_PER_COUNTRY)
    results[country] = got
    
    # Save progress after each country
    with open(os.path.join(DRIVE_DIR, 'progress.json'), 'w') as f:
        json.dump({'results': results, 'total': scraper.count, 'timestamp': str(datetime.now())}, f, indent=2)

clear_output(wait=True)
print('='*60)
print('🏁 ALL COUNTRIES COMPLETE!')
print('='*60)
print(f'Total files downloaded: {scraper.count}')
print(f'Location: {DRIVE_DIR}')
print()
for country, got in sorted(results.items(), key=lambda x: -x[1]):
    if got > 0:
        print(f'  {country.replace("_"," ").title():25s} : {got} files')

In [ ]:
# Step 5: Summary
import glob
files = glob.glob(os.path.join(DRIVE_DIR, '*.pptx')) + glob.glob(os.path.join(DRIVE_DIR, '*.ppt'))
total_gb = sum(os.path.getsize(f) for f in files) / (1024**3)
print(f'📊 Final Summary:')
print(f'   Total clean files: {len(files)}')
print(f'   Total size: {total_gb:.2f} GB')
print(f'   Location: Google Drive > PPTX_Global_Download')